# Build and debug a PyPSA network
Input time series and dispatch outputs are distinct. No real LUT equivalence is implied.

In [ ]:
from pathlib import Path
from lut_to_pypsa.pipeline import load_tables, build_network
data_dir = Path("../data/synthetic") if Path("../data/synthetic").exists() else Path("data/synthetic")
n, report = build_network(*load_tables(data_dir))
print(report)
display(n.buses, n.generators, n.loads_t.p_set.head())

In [ ]:
assert n.loads_t.p_set.notna().all().all()
assert set(n.generators.bus).issubset(n.buses.index)
print("Input checks passed; generator dispatch is not available until optimize() runs")

In [ ]:
try:
    import highspy
except ImportError:
    print("Install highspy to solve: %pip install highspy")
else:
    status, condition = n.optimize(solver_name="highs")
    print(status, condition)
    assert status == "ok" and condition == "optimal"
    display(n.generators_t.p.head())